# 08 - Final Validation and Report

Notebook ini melakukan validasi akhir terhadap seluruh dataset processed dan menyusun laporan ringkas hasil wrangling.

Validasi akhir diperlukan untuk memastikan seluruh output antar-notebook konsisten, terutama pada relasi antar tabel yang saling bergantung.

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Mengatur tampilan dataframe agar output notebook lebih mudah dibaca.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Cari root project otomatis
# Mengambil lokasi kerja notebook saat ini.
current_path = Path.cwd().resolve()

# Menelusuri parent folder sampai menemukan root project yang memiliki folder data/raw.
for path in [current_path] + list(current_path.parents):
    if (path / "data" / "raw").exists():
        PROJECT_ROOT = path
        break

# Menentukan folder sumber data raw.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
# Menentukan folder output data hasil cleaning.
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# Menentukan folder output report dan validation summary.
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"

# Membuat folder processed jika belum tersedia.
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# Membuat folder reports jika belum tersedia.
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Menampilkan path project untuk memastikan notebook membaca folder yang benar.
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_DIR      :", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("REPORT_DIR   :", REPORT_DIR)

PROJECT_ROOT : C:\Data Codingan\student_stress_data_science
RAW_DIR      : C:\Data Codingan\student_stress_data_science\data\raw
PROCESSED_DIR: C:\Data Codingan\student_stress_data_science\data\processed
REPORT_DIR   : C:\Data Codingan\student_stress_data_science\outputs\reports


## 1. Load Processed Dataset

In [ ]:
# menjalankan bagian kode pada tahap ini sesuai konteks notebook.
# Membaca file CSV ke dalam dataframe.
users_clean = pd.read_csv(PROCESSED_DIR / "users_clean.csv")
authentications_clean = pd.read_csv(PROCESSED_DIR / "authentications_clean.csv")
daily_clean = pd.read_csv(PROCESSED_DIR / "daily_activities_clean.csv")
stress_predictions_clean = pd.read_csv(PROCESSED_DIR / "stress_predictions_clean.csv")
weekly_summaries_clean = pd.read_csv(PROCESSED_DIR / "weekly_summaries_clean.csv")
recommendations_clean = pd.read_csv(PROCESSED_DIR / "recommendations_clean.csv")
insights_clean = pd.read_csv(PROCESSED_DIR / "insights_clean.csv")

processed_files = {
    "users_clean.csv": users_clean,
    "authentications_clean.csv": authentications_clean,
    "daily_activities_clean.csv": daily_clean,
    "stress_predictions_clean.csv": stress_predictions_clean,
    "weekly_summaries_clean.csv": weekly_summaries_clean,
    "recommendations_clean.csv": recommendations_clean,
    "insights_clean.csv": insights_clean,
}

for name, df in processed_files.items():
    print(name, df.shape)

users_clean.csv (300, 7)
authentications_clean.csv (300, 6)
daily_activities_clean.csv (26984, 18)
stress_predictions_clean.csv (26579, 7)
weekly_summaries_clean.csv (3600, 13)
recommendations_clean.csv (26784, 10)
insights_clean.csv (29551, 7)


## 2. Final Validation

In [ ]:
# membuat tabel validasi untuk memastikan hasil cleaning memenuhi aturan kualitas data.
# Membuat ringkasan validasi akhir lintas tabel.
validation_summary = pd.DataFrame([
    {"rule": "users.id unique", "passed": users_clean["id"].is_unique},
    {"rule": "authentications.user_id exists in users", "passed": set(authentications_clean["user_id"]).issubset(set(users_clean["id"]))},
    {"rule": "daily_activities.id unique", "passed": daily_clean["id"].is_unique},
    # Mengecek keberadaan data duplicate berdasarkan aturan yang relevan.
    {"rule": "daily_activities user_id + activity_date unique", "passed": not daily_clean.duplicated(["user_id", "activity_date"]).any()},
    {"rule": "social_media_hours <= screen_time_hours", "passed": (daily_clean["social_media_hours"] <= daily_clean["screen_time_hours"]).all()},
    {"rule": "stress_predictions.activity_id exists in daily_activities", "passed": set(stress_predictions_clean["activity_id"]).issubset(set(daily_clean["id"]))},
    {"rule": "stress_predictions.activity_id unique", "passed": stress_predictions_clean["activity_id"].is_unique},
    {"rule": "stress_score 0-100", "passed": stress_predictions_clean["stress_score"].between(0, 100).all()},
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    {"rule": "stress_level valid", "passed": stress_predictions_clean["stress_level"].isin(["Low", "Medium", "High"]).all()},
    # Mengecek keberadaan data duplicate berdasarkan aturan yang relevan.
    {"rule": "weekly_summaries user_id + week_start + week_end unique", "passed": not weekly_summaries_clean.duplicated(["user_id", "week_start", "week_end"]).any()},
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    {"rule": "recommendations period_type valid", "passed": recommendations_clean["period_type"].isin(["daily", "weekly"]).all()},
    {"rule": "insights period_type valid", "passed": insights_clean["period_type"].isin(["daily", "weekly"]).all()},
])

validation_summary

,rule,passed
0,users.id unique,True
1,authentications.user_id exists in users,True
2,daily_activities.id unique,True
3,daily_activities user_id + activity_date unique,True
4,social_media_hours <= screen_time_hours,True
5,stress_predictions.activity_id exists in daily...,True
6,stress_predictions.activity_id unique,True
7,stress_score 0-100,True
8,stress_level valid,True
9,weekly_summaries user_id + week_start + week_e...,True


## 3. Domino Effect Audit

In [ ]:
# memeriksa dampak relasi antar tabel setelah proses cleaning.
daily_without_prediction = daily_clean[
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    ~daily_clean["id"].isin(stress_predictions_clean["activity_id"])
]

prediction_without_daily = stress_predictions_clean[
    # Memvalidasi apakah nilai kolom berada dalam daftar nilai yang diperbolehkan.
    ~stress_predictions_clean["activity_id"].isin(daily_clean["id"])
]

# Membuat audit efek relasi antar tabel setelah cleaning.
domino_audit = pd.DataFrame([
    {"check": "daily activities without prediction", "count": len(daily_without_prediction)},
    {"check": "predictions without daily activity", "count": len(prediction_without_daily)},
])

domino_audit

,check,count
0,daily activities without prediction,405
1,predictions without daily activity,0


## Insight:

Dataset master hasil cleaning tidak harus memiliki jumlah baris yang sama antar tabel. Tabel aktivitas harian dapat memiliki baris yang tidak memiliki prediction valid. Untuk kebutuhan analisis berbasis target, sinkronisasi dilakukan pada tahap pembentukan dataset analisis menggunakan inner join.

Pendekatan ini menjaga data master tetap lengkap sekaligus memastikan dataset analisis hanya berisi baris yang memiliki label target.

## 4. Helper Function untuk Report Markdown

In [ ]:
# mendefinisikan fungsi bantu lokal yang digunakan pada notebook ini.
def dataframe_to_markdown(df):
    # Membuat tabel markdown sederhana tanpa dependency tabulate.
    df = df.copy().astype(str)

    header = "| " + " | ".join(df.columns) + " |"
    separator = "| " + " | ".join(["---"] * len(df.columns)) + " |"

    rows = []
    for _, row in df.iterrows():
        rows.append("| " + " | ".join(row.values) + " |")

    return "\n".join([header, separator] + rows)

## 5. Save Final Report

In [ ]:
# membuat tabel validasi untuk memastikan hasil cleaning memenuhi aturan kualitas data.
# Menyimpan dataframe ke file CSV.
validation_summary.to_csv(REPORT_DIR / "validation_summary.csv", index=False)
domino_audit.to_csv(REPORT_DIR / "domino_effect_audit.csv", index=False)

row_count = pd.DataFrame([
    {"dataset": name, "rows": len(df)}
    for name, df in processed_files.items()
])

report = "# Data Wrangling Final Report\n\n"

report += "## 1. Row Count Result\n\n"
report += dataframe_to_markdown(row_count)
report += "\n\n"

report += "## 2. Validation Summary\n\n"
report += dataframe_to_markdown(validation_summary)
report += "\n\n"

report += "## 3. Domino Effect Audit\n\n"
report += dataframe_to_markdown(domino_audit)
report += "\n\n"

report += "## 4. Important Note\n\n"
report += "- Wrangling dilakukan modular per dataset.\n"
report += "- Tidak ada perhitungan ulang stress_score.\n"
report += "- Tidak ada pembuatan ulang stress_level.\n"
report += "- Dataset modelling dibuat terpisah menggunakan inner join.\n"

# Menulis report dalam format markdown ke folder reports.
(REPORT_DIR / "data_wrangling_final_report.md").write_text(report, encoding="utf-8")

print("Saved:", REPORT_DIR / "data_wrangling_final_report.md")

Saved: C:\Data Codingan\student_stress_data_science\outputs\reports\data_wrangling_final_report.md


# 6. Conclusion and EDA Considerations

## 6.1 Conclusion Hasil Data Wrangling

Berdasarkan seluruh proses data wrangling yang telah dilakukan, setiap dataset telah melewati tahapan pemeriksaan, pembersihan, validasi, dan penyimpanan ke folder `data/processed/`. Proses wrangling dilakukan secara modular agar setiap tabel dapat dianalisis berdasarkan fungsi dan karakteristik datanya masing-masing.

Secara umum, hasil wrangling menunjukkan bahwa struktur data sudah cukup siap untuk digunakan pada tahap analisis lanjutan. Tabel `users`, `authentications`, `weekly_summaries`, `recommendations`, dan `insights` lebih banyak membutuhkan standardisasi format, validasi relasi, dan validasi kategori. Sementara itu, tabel `daily_activities` menjadi dataset dengan proses cleaning paling intensif karena merepresentasikan input aktivitas harian pengguna yang paling rentan terhadap missing value, format tidak konsisten, duplikasi pengisian, dan nilai di luar batas logis.

Dataset `daily_activities_clean` telah dibersihkan dengan mempertahankan `id` asli dari data raw. Keputusan ini penting karena `id` pada tabel tersebut digunakan sebagai referensi oleh `stress_predictions.activity_id`. Dengan demikian, proses cleaning tidak merusak struktur relasi antar tabel.

Tabel `stress_predictions_clean` telah divalidasi agar hanya menyimpan prediction yang memiliki relasi valid terhadap `daily_activities_clean`. Jika terdapat daily activity yang tidak memiliki pasangan prediction, data tersebut tetap dipertahankan sebagai bagian dari data aktivitas bersih. Namun, baris tersebut tidak digunakan untuk analisis atau modelling yang membutuhkan target `stress_level`.

Tabel `recommendations_clean` dan `insights_clean` juga telah divalidasi berdasarkan aturan source harian dan mingguan. Missing value pada salah satu source id tidak langsung dianggap sebagai kesalahan, karena struktur tabel memang mendukung dua jenis sumber data, yaitu daily dan weekly.

Dengan demikian, hasil akhir wrangling dapat disimpulkan sebagai berikut:

- Dataset processed sudah memenuhi aturan dasar terkait key, relasi, kategori, dan rentang nilai.
- Dataset aktivitas harian sudah siap digunakan sebagai sumber fitur utama.
- Dataset prediction sudah siap digunakan sebagai sumber target berbasis `stress_level`.
- Dataset mingguan dapat digunakan untuk analisis tren dan ringkasan monitoring.
- Dataset recommendations dan insights dapat digunakan sebagai data output pendukung, bukan sebagai sumber fitur utama untuk modelling.

## 6.2 Pertimbangan untuk Tahap EDA

Tahap EDA perlu difokuskan pada hubungan antara aktivitas harian dan tingkat stres. Oleh karena itu, EDA utama tidak sebaiknya dilakukan langsung pada seluruh tabel secara merata, karena setiap tabel memiliki fungsi yang berbeda dalam sistem.

Dataset utama untuk EDA harian adalah hasil penggabungan antara:

- `daily_activities_clean.csv`
- `stress_predictions_clean.csv`

Penggabungan dilakukan menggunakan inner join dengan relasi:

```python
daily_activities_clean.id = stress_predictions_clean.activity_id